# Lesson 05 — Polynomial + Parameter Selection

<h2 dir="rtl">פרק 1 — פתיחה ורענון משיעור 04</h2>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

<h2 dir="rtl">פרק 2 — דאטה ומודל לינארי ראשוני</h2>

In [ ]:
rng = np.random.default_rng(42)
n = 40
x = rng.uniform(-3, 3, size=n)
eps = rng.normal(0, 1.5, size=n)
y = 3 + 2*x - 0.5*x**2 + eps

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(x, y, alpha=0.7)
ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
ax[0].set_title('Scatter')

X1 = sm.add_constant(pd.DataFrame({'x': x}))
res1 = sm.OLS(y, X1).fit()
ax[1].scatter(res1.fittedvalues, res1.resid, alpha=0.7)
ax[1].axhline(0, color='red', linestyle='--')
ax[1].set_xlabel('fitted'); ax[1].set_ylabel('residuals')
ax[1].set_title(f'Residuals  (linear fit, R^2 = {res1.rsquared:.3f})')
plt.tight_layout(); plt.show()

<h2 dir="rtl">פרק 3 — פולינומיאלי: דרגות 1, 2, 5</h2>

In [ ]:
X_poly = pd.DataFrame({f'x{d}': x**d for d in range(1, 6)})

fits = {}
for d in [1, 2, 5]:
    cols = [f'x{i}' for i in range(1, d+1)]
    X = sm.add_constant(X_poly[cols])
    fits[d] = sm.OLS(y, X).fit()

for d, res in fits.items():
    print(f'degree {d}: R^2 = {res.rsquared:.4f}, k = {len(res.params)}')

In [ ]:
xs = np.linspace(-3.2, 3.2, 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.5, label='data')
for d, color in zip([1, 2, 5], ['red', 'green', 'blue']):
    cols = [f'x{i}' for i in range(1, d+1)]
    Xs = sm.add_constant(pd.DataFrame({c: xs**i for c, i in zip(cols, range(1, d+1))}))
    ax.plot(xs, fits[d].predict(Xs), color=color, label=f'degree {d}', linewidth=2)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend()
plt.show()

<h2 dir="rtl">פרק 4 — Train vs. Test — overfitting בפועל</h2>

In [ ]:
idx = rng.permutation(n)
split = int(n * 0.7)
tr, te = idx[:split], idx[split:]
y_tr, y_te = y[tr], y[te]

rows = []
for d in [1, 2, 3, 4, 5]:
    cols = [f'x{i}' for i in range(1, d+1)]
    X_tr = sm.add_constant(X_poly[cols].iloc[tr])
    X_te = sm.add_constant(X_poly[cols].iloc[te])
    res = sm.OLS(y_tr, X_tr).fit()
    yhat_te = res.predict(X_te)
    ss_res = ((y_te - yhat_te)**2).sum()
    ss_tot = ((y_te - y_te.mean())**2).sum()
    test_r2 = 1 - ss_res / ss_tot
    test_rmse = np.sqrt(ss_res / len(y_te))
    rows.append({'degree': d, 'train_R2': res.rsquared,
                 'test_R2': test_r2, 'test_RMSE': test_rmse})
pd.DataFrame(rows).round(4)

<h2 dir="rtl">פרק 5 — מעבר: בחירת מודל</h2>

<h2 dir="rtl">פרק 6 — AIC, BIC, adjusted-R²</h2>

In [ ]:
rows = []
for d in [1, 2, 3, 4, 5]:
    cols = [f'x{i}' for i in range(1, d+1)]
    X = sm.add_constant(X_poly[cols])
    res = sm.OLS(y, X).fit()
    rows.append({'degree': d,
                 'R2': res.rsquared,
                 'adjR2': res.rsquared_adj,
                 'AIC': res.aic,
                 'BIC': res.bic})
df_crit = pd.DataFrame(rows)
df_crit.round(4)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].plot(df_crit['degree'], df_crit['AIC'], 'o-', color='blue')
axs[0].set_xlabel('degree'); axs[0].set_ylabel('AIC')
axs[0].set_title('AIC (lower is better)')
axs[1].plot(df_crit['degree'], df_crit['BIC'], 'o-', color='red')
axs[1].set_xlabel('degree'); axs[1].set_ylabel('BIC')
axs[1].set_title('BIC (lower is better)')
axs[2].plot(df_crit['degree'], df_crit['adjR2'], 'o-', color='green')
axs[2].set_xlabel('degree'); axs[2].set_ylabel('adjR²')
axs[2].set_title('adjR² (higher is better)')
plt.tight_layout(); plt.show()

<h2 dir="rtl">פרק 7 — Stepwise (forward selection)</h2>

In [ ]:
def forward_selection(y, candidates, threshold=0.05, verbose=True):
    """Forward: at each step, add the variable with lowest p-value if < threshold."""
    included = []
    while True:
        excluded = [c for c in candidates.columns if c not in included]
        if not excluded:
            break
        best_pval, best_feat = 1.0, None
        for f in excluded:
            X = sm.add_constant(candidates[included + [f]])
            res = sm.OLS(y, X).fit()
            if res.pvalues[f] < best_pval:
                best_pval, best_feat = res.pvalues[f], f
        if best_pval < threshold:
            included.append(best_feat)
            if verbose:
                print(f'  Add  {best_feat:<4} (p = {best_pval:.4f})')
        else:
            if verbose:
                print(f'  Stop (best p = {best_pval:.4f} >= {threshold})')
            break
    return included

print('=== Forward Selection ===')
fwd = forward_selection(y, X_poly)
print(f'\nFinal: {fwd}')

In [ ]:
def backward_elimination(y, candidates, threshold=0.05, verbose=True):
    """Backward: start with all, drop variable with highest p-value if > threshold."""
    included = list(candidates.columns)
    while True:
        X = sm.add_constant(candidates[included])
        res = sm.OLS(y, X).fit()
        pvals = res.pvalues.iloc[1:]   # skip the intercept
        worst_pval = pvals.max()
        if worst_pval > threshold:
            worst_feat = pvals.idxmax()
            included.remove(worst_feat)
            if verbose:
                print(f'  Drop {worst_feat:<4} (p = {worst_pval:.4f})')
        else:
            if verbose:
                print(f'  Stop (max p = {worst_pval:.4f} <= {threshold})')
            break
    return included

print('=== Backward Elimination ===')
bwd = backward_elimination(y, X_poly)
print(f'\nFinal: {bwd}')

In [ ]:
def stepwise_selection(y, candidates, threshold_in=0.05, threshold_out=0.05, verbose=True):
    """Stepwise: combine forward and backward at each iteration."""
    included = []
    while True:
        changed = False
        # Forward step
        excluded = [c for c in candidates.columns if c not in included]
        if excluded:
            best_pval, best_feat = 1.0, None
            for f in excluded:
                X = sm.add_constant(candidates[included + [f]])
                res = sm.OLS(y, X).fit()
                if res.pvalues[f] < best_pval:
                    best_pval, best_feat = res.pvalues[f], f
            if best_pval < threshold_in:
                included.append(best_feat)
                changed = True
                if verbose:
                    print(f'  Add  {best_feat:<4} (p = {best_pval:.4f})')
        # Backward step
        if included:
            X = sm.add_constant(candidates[included])
            res = sm.OLS(y, X).fit()
            pvals = res.pvalues.iloc[1:]
            worst_pval = pvals.max()
            if worst_pval > threshold_out:
                worst_feat = pvals.idxmax()
                included.remove(worst_feat)
                changed = True
                if verbose:
                    print(f'  Drop {worst_feat:<4} (p = {worst_pval:.4f})')
        if not changed:
            break
    return included

print('=== Stepwise (combined) ===')
sw = stepwise_selection(y, X_poly)
print(f'\nFinal: {sw}')

<h2 dir="rtl">פרק 8 — סגירה</h2>

### Write your answer here.

<div dir="rtl">יש לכם שני מודלים מאומנים על אותו דאטה: מודל פולינומיאלי <code>degree 5</code> עם <code>R² = 0.91</code> ו-<code>AIC = 128</code>, ומודל פולינומיאלי <code>degree 2</code> עם <code>R² = 0.91</code> ו-<code>AIC = 122</code>. <b>איזה מהם תבחרו, ולמה?</b> איזה מהם ה-<code>summary()</code> יציג כ"יותר מובהק" (יותר מקדמים עם <i>p</i> קטן)? למה זה לא הקריטריון הנכון?</div>